In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from cmm_eval import BatchRecorder, BatchStore, CMMSaturationEvaluator, default_checkpoints, plot_sweep

import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import seed_everything

seed_everything(42)

In [2]:
d,h,l = 2048, 4, 5
chunk_length =  (h ** l)

compositionOp='SCC'#'JLT' #'ProdWithShuffling' #

NUMBER_OF_DOCS=500

CLEN_AFTER_RUN = True

In [3]:
padding_vector_component_values= {
    'ProdWithShuffling': 1/(d**(1/2)),
    'Prod': 1/(d**(1/2)), 
    'Sum': 0,
    'SCC': 'circular_convolution_identity',
    'JLT': 'in_distribution'
}
padding_vector_component_values = padding_vector_component_values[compositionOp]


# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
    hidden_size=d, 
    num_hidden_layers=l,
    num_attention_heads=h,
    chunk_length=chunk_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    compositionOp=compositionOp,#'ProdWithShuffling',
    padding_seq_idx=tokenizer.pad_token_id,
    padding_vector_component_values=padding_vector_component_values #new!
                    
)

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 
model.training = True
model.train()
print(model)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Setting pad token and pad token id = <|endoftext|>, 0
MeMoEmbedding circular_convolution_identity
MeMoEmbedding 0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMoForCausalLM(
  (memo): MeMo(
    (encoder): MeMoEmbedding(50277, 2048, padding_idx=0)
    (output_encoder): MeMoEmbedding(50277, 2048, padding_idx=0)
    (layers): MeMoLayers(
      (0-4): 5 x MeMoLayer(
        (CMM_OUT): CorrelationMatrixMemory(in_features=2048, out_features=2048)
      )
    )
  )
  (lm_head): MeMoEmbedding(50277, 2048, padding_idx=0)
)
GPU: NVIDIA RTX A6000 is available.


In [4]:
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from cmm_eval import (BatchRecorder, BatchStore, CMMSaturationEvaluator,
                      default_checkpoints, plot_sweep)


class CMM_evaluator:
    """
    Launches the CMM saturation experiments.

        exp = CMM_evaluator(model, tokenizer, "memo_dump")
        exp.record(data)                 # once: runs model.memorize_text and dumps every layer's memorize() calls
        exp.evaluate(2)                  # one layer
        exp.evaluate_all()               # every layer

    In a new process (model rebuilt from scratch), call exp.load_layer_states() before evaluating, so that the
    random projections / shuffling buffers are the ones used when recording.
    """

    def __init__(self, model, tokenizer, dump_dir):
        self.model = model
        self.tokenizer = tokenizer  # not used here, kept for compatibility
        self.dump_dir = Path(dump_dir)

    # ------------------------------------------------------------------ #
    # recording
    # ------------------------------------------------------------------ #
    def record(self, data, max_batches_per_layer=1000, save_dtype=None,
               append=False, save_layer_states=True):
        """
        Run model.memorize_text on every item of `data` while recording each layer's memorize() calls.
        max_batches_per_layer caps disk usage; save_dtype=torch.float16 halves it (replay is then lossy).
        Refuses to write into a folder that already holds batches unless append=True, because
        recording twice silently duplicates the data.
        """
        if not append and any(self.dump_dir.glob("layer_*/manifest.jsonl")):
            raise FileExistsError(
                f"{self.dump_dir} already contains recorded batches; "
                "use a new dump_dir or pass append=True")
        self.dump_dir.mkdir(parents=True, exist_ok=True)

        rec = BatchRecorder(self.dump_dir, save_dtype=save_dtype,
                            max_batches_per_layer=max_batches_per_layer)
        for l, layer in enumerate(self.model.memo.layers):
            rec.attach(layer, l)
        try:
            for i in range(len(data)):
                self.model.memorize_text(data[i])
        finally:
            rec.detach_all()  # always restore the original memorize, even if something fails

        if save_layer_states:
            self.save_layer_states()
        return self.summary()

    def summary(self):
        """Number of recorded batches and pairs per layer."""
        out = {}
        for l in range(len(self.model.memo.layers)):
            try:
                store = BatchStore(self.dump_dir, l)
            except FileNotFoundError:
                continue
            out[l] = dict(n_batches=len(store),
                          n_pairs=sum(store.n_pairs(i) for i in range(len(store))))
        return out

    # ------------------------------------------------------------------ #
    # layer states (projections / shuffling buffers must match the recording)
    # ------------------------------------------------------------------ #
    def save_layer_states(self):
        torch.save({l: layer.state_dict() for l, layer in enumerate(self.model.memo.layers)},
                   self.dump_dir / "layers.pt")

    def load_layer_states(self):
        states = torch.load(self.dump_dir / "layers.pt", map_location="cpu")
        for l, layer in enumerate(self.model.memo.layers):
            layer.load_state_dict(states[l])

    # ------------------------------------------------------------------ #
    # evaluation
    # ------------------------------------------------------------------ #
    def evaluate(self, layer_level, n_holdout_batches=2, n_points=8, n_eval=2000, n_fresh=2000,
                 decode=True, prior_smoothing=1.0, plot=True):
        """
        Replay the recorded batches of one layer into an empty copy of that layer and measure recall
        (cosine, top-1, held-out crosstalk) and, if decode=True, the decoders (dot / Bayes temperature /
        fitted temperature, with and without prior). Returns the list of per-checkpoint rows.
        """
        # sweep() zeroes CMM_OUT, so work on a copy: the model's real memory stays untouched.
        layer = copy.deepcopy(self.model.memo.layers[layer_level])
        store = BatchStore(self.dump_dir, layer_level)

        ev = CMMSaturationEvaluator(layer, store, n_holdout_batches=n_holdout_batches)
        checkpoints = default_checkpoints(ev.max_train_batches, n_points=n_points)

        rows = ev.sweep(checkpoints, n_eval=n_eval, n_fresh=n_fresh,
                        out_json=str(self.dump_dir / f"layer{layer_level}_sweep.json"),
                        decode=decode, prior_smoothing=prior_smoothing)
        if plot:
            print(f"Plotting layer {layer_level} sweep to {self.dump_dir / f'layer{layer_level}_sweep.png'}")
            fig = plot_sweep(rows, str(self.dump_dir / f"layer{layer_level}_sweep.png"))
            plt.close(fig)

        del ev, layer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return rows

    def evaluate_all(self, layers=None, **kwargs):
        """evaluate() on every layer (or on `layers`); kwargs are passed through. Returns {layer: rows}."""
        layers = range(len(self.model.memo.layers)) if layers is None else layers
        return {l: self.evaluate(l, **kwargs) for l in layers}

In [5]:
format(NUMBER_OF_DOCS, '06d')

'000500'

In [6]:
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk, concatenate_datasets

data = load_from_disk(f'LearningEvaluation/training_data/scaling_curve/n={format(NUMBER_OF_DOCS, "06d")}') #original_training_data

tok_data = data.map(lambda x: tokenizer.get_text_batch_encoding(x['text'], truncation=True))
tok_data.set_format(type='torch', columns=['input_ids', 'labels'])
print(tok_data)
tok_data[0]['input_ids'].shape, tok_data[0]['labels'].shape

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'labels'],
    num_rows: 500
})


(torch.Size([1, 1024]), torch.Size([1, 1024]))

In [7]:
from torch import exp


dump_dir = f'memo_dump_v2_{compositionOp}_{h}|{l}|{d}_{format(NUMBER_OF_DOCS, "06d")}'


cmm_evaluator = CMM_evaluator(
    model=model, 
    tokenizer=tokenizer, 
    dump_dir=dump_dir
)

cmm_evaluator.record(tok_data)

{0: {'n_batches': 500, 'n_pairs': 512000},
 1: {'n_batches': 500, 'n_pairs': 510000},
 2: {'n_batches': 500, 'n_pairs': 504000},
 3: {'n_batches': 500, 'n_pairs': 480000},
 4: {'n_batches': 500, 'n_pairs': 384000}}

In [8]:
results = cmm_evaluator.evaluate_all()

for layer_level in range(l):
    print("*"*80)
    print(layer_level)
    print("*"*80)

    display(results[layer_level])

/home/elena/MeMo/LearningEvaluation/cmm_eval.py:223: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(self.dir / f"batch_{i:06d}.keys.pt", map_location="cpu")
/h

batches=    1 P=    1024 alpha=0.500  cos=0.784 (theory 0.816, p05 0.614)  top1=0.954  d_eff~2771/2048  |k|=1.00
      NLL dot=9.21 bayes=69.09 bayes+prior=69.08 fit+prior=8.63 (uniform 10.28, prior-only 8.15)  top1 fit+prior=0.954  beta_fit/theory=0.12  V=29045
batches=    2 P=    2048 alpha=1.000  cos=0.671 (theory 0.707, p05 0.511)  top1=0.951  d_eff~4723/2048  |k|=1.00
      NLL dot=9.06 bayes=89.10 bayes+prior=89.10 fit+prior=11.14 (uniform 10.28, prior-only 8.14)  top1 fit+prior=0.951  beta_fit/theory=0.12  V=29045
batches=    6 P=    6144 alpha=3.000  cos=0.483 (theory 0.500, p05 0.329)  top1=0.916  d_eff~3345/2048  |k|=1.00
      NLL dot=9.18 bayes=19.65 bayes+prior=19.74 fit+prior=2.57 (uniform 10.28, prior-only 7.89)  top1 fit+prior=0.909  beta_fit/theory=0.12  V=29045
batches=   14 P=   14336 alpha=7.000  cos=0.342 (theory 0.354, p05 0.188)  top1=0.740  d_eff~4725/2048  |k|=1.00
      NLL dot=9.10 bayes=33.53 bayes+prior=34.26 fit+prior=5.11 (uniform 10.28, prior-only 7.82) 

[{'n_batches': 1,
  'P': 1024,
  'alpha': 0.5,
  'cos_mean': 0.7844977378845215,
  'cos_p05': 0.6143152117729187,
  'cos_min': 0.2244178056716919,
  'cos_theory': 0.8164965809277261,
  'rel_err': 0.8684667944908142,
  'rel_err_theory': 0.7071067811865476,
  'top1_acc': 0.9541015625,
  'top1_chance': 0.0009765625,
  'key_norm_mean': 1.0,
  'power_ratio': 3.706505537033081,
  'd_eff_est': 2771.461669921875,
  'fro2_per_pair': 1.063240242567452,
  'dec': {'n_candidates': 29045,
   'n_distinct_stored': 375,
   'gamma': 1.0,
   'beta_theory': 4066.7537719994693,
   'beta_fit': 508.34420776367205,
   'beta_fit_prior': 508.34420776367205,
   'nll_uniform': 10.276601630426633,
   'top1_chance': 3.442933379239112e-05,
   'prior_only': {'nll': 8.154419898986816,
    'top1': 0.0859375,
    'top5': 0.244140625},
   'info_from_memory_nats': -0.4801807403564453,
   'decoders': {'dot_T1': {'nll': 9.205273628234863,
     'top1': 0.9541015625,
     'top5': 0.998046875},
    'bayes_T': {'nll': 69.086898

********************************************************************************
1
********************************************************************************


[{'n_batches': 1,
  'P': 1020,
  'alpha': 0.498046875,
  'cos_mean': 0.8177688717842102,
  'cos_p05': 0.7485051155090332,
  'cos_min': 0.3913719058036804,
  'cos_theory': 0.8170286739010644,
  'rel_err': 0.7039825320243835,
  'rel_err_theory': 0.7057243619147634,
  'top1_acc': 1.0,
  'top1_chance': 0.000980392156862745,
  'key_norm_mean': 1.0,
  'power_ratio': 3.024193286895752,
  'd_eff_est': 2064.677152633667,
  'fro2_per_pair': 1.0007410997199468,
  'dec': {'n_candidates': 29017,
   'n_distinct_stored': 374,
   'gamma': 1.0,
   'beta_theory': 4082.37813106817,
   'beta_fit': 32659.025390624978,
   'beta_fit_prior': 26527.371093750015,
   'nll_uniform': 10.275637144113018,
   'top1_chance': 3.4462556432436155e-05,
   'prior_only': {'nll': 8.157383918762207,
    'top1': 0.08627451211214066,
    'top5': 0.24411766231060028},
   'info_from_memory_nats': 8.15753171959659,
   'decoders': {'dot_T1': {'nll': 9.267069816589355, 'top1': 1.0, 'top5': 1.0},
    'bayes_T': {'nll': -2.14221436181

********************************************************************************
2
********************************************************************************


[{'n_batches': 1,
  'P': 1008,
  'alpha': 0.4921875,
  'cos_mean': 0.8204861879348755,
  'cos_p05': 0.7567946314811707,
  'cos_min': 0.6266285181045532,
  'cos_theory': 0.8186312161546413,
  'rel_err': 0.7024518251419067,
  'rel_err_theory': 0.701560760020114,
  'top1_acc': 1.0,
  'top1_chance': 0.000992063492063492,
  'key_norm_mean': 1.0,
  'power_ratio': 3.050915241241455,
  'd_eff_est': 2067.3225631713867,
  'fro2_per_pair': 1.0080935212056525,
  'dec': {'n_candidates': 28925,
   'n_distinct_stored': 367,
   'gamma': 1.0,
   'beta_theory': 4130.711612569277,
   'beta_fit': 4130.711425781252,
   'beta_fit_prior': 4130.711425781252,
   'nll_uniform': 10.272461552061877,
   'top1_chance': 3.457216940363008e-05,
   'prior_only': {'nll': 8.149190902709961,
    'top1': 0.085317462682724,
    'top5': 0.245039701461792},
   'info_from_memory_nats': 8.14893792165094,
   'decoders': {'dot_T1': {'nll': 9.256403923034668, 'top1': 1.0, 'top5': 1.0},
    'bayes_T': {'nll': -1.3200064131524414e-0

********************************************************************************
3
********************************************************************************


[{'n_batches': 1,
  'P': 960,
  'alpha': 0.46875,
  'cos_mean': 0.8266558647155762,
  'cos_p05': 0.7628391981124878,
  'cos_min': 0.597752571105957,
  'cos_theory': 0.8251369970070347,
  'rel_err': 0.6784944534301758,
  'rel_err_theory': 0.6846531968814576,
  'top1_acc': 1.0,
  'top1_chance': 0.0010416666666666667,
  'key_norm_mean': 1.0,
  'power_ratio': 3.1066296100616455,
  'd_eff_est': 2022.3644256591797,
  'fro2_per_pair': 0.9961955265901714,
  'dec': {'n_candidates': 28506,
   'n_distinct_stored': 349,
   'gamma': 1.0,
   'beta_theory': 4337.94258413499,
   'beta_fit': 4337.942382812497,
   'beta_fit_prior': 12269.553733047378,
   'nll_uniform': 10.257869870414977,
   'top1_chance': 3.508033396477934e-05,
   'prior_only': {'nll': 8.132111549377441,
    'top1': 0.08750000596046448,
    'top5': 0.25208336114883423},
   'info_from_memory_nats': 8.132090306280588,
   'decoders': {'dot_T1': {'nll': 9.253911018371582, 'top1': 1.0, 'top5': 1.0},
    'bayes_T': {'nll': -5.364418029785156

********************************************************************************
4
********************************************************************************


[{'n_batches': 1,
  'P': 768,
  'alpha': 0.375,
  'cos_mean': 0.8543195724487305,
  'cos_p05': 0.8036925196647644,
  'cos_min': 0.5147685408592224,
  'cos_theory': 0.8528028654224417,
  'rel_err': 0.6082993745803833,
  'rel_err_theory': 0.6123724356957945,
  'top1_acc': 0.9986979365348816,
  'top1_chance': 0.0013020833333333333,
  'key_norm_mean': 1.0,
  'power_ratio': 3.630398988723755,
  'd_eff_est': 2020.1464233398438,
  'fro2_per_pair': 0.9997668184967119,
  'dec': {'n_candidates': 26526,
   'n_distinct_stored': 295,
   'gamma': 1.0,
   'beta_theory': 5423.357384224459,
   'beta_fit': 43386.859374999985,
   'beta_fit_prior': 35241.0806573315,
   'nll_uniform': 10.185880663054299,
   'top1_chance': 3.769886149438287e-05,
   'prior_only': {'nll': 8.135421752929688,
    'top1': 0.0911458358168602,
    'top5': 0.26171875},
   'info_from_memory_nats': 8.135804494231706,
   'decoders': {'dot_T1': {'nll': 9.178208351135254,
     'top1': 0.9986979365348816,
     'top5': 1.0},
    'bayes_T'

In [9]:
import shutil

if True:
    for layer_level in range(l):
        shutil.rmtree(f"{dump_dir}/layer_{layer_level}")

In [10]:
#exit()